In [1]:
import biotite.structure.io as bsio
import biotite.structure as struc
print("Everything is working!")

Everything is working!


In [2]:
import urllib.request
import biotite.structure.io as bsio
import biotite.structure as struc

# 1. Download the PDB file directly using standard Python libraries
url = "https://files.rcsb.org/download/1M17.pdb"
urllib.request.urlretrieve(url, "1M17.pdb")
print("Downloaded 1M17.pdb successfully!")

# 2. Load it using Biotite
source_array = bsio.load_structure("1M17.pdb", extra_fields=["charge"])

# 3. Verification
print(f"Successfully loaded structure! Total atoms: {len(source_array)}")

Downloaded 1M17.pdb successfully!
Successfully loaded structure! Total atoms: 2546


In [3]:
# Filter for heteroatoms that are not water molecules ('HOH')
ligand_mask = (source_array.hetero == True) & (source_array.res_name != "HOH")
ligand_array = source_array[ligand_mask]

# Verification: Check what ligand residue names are present
unique_residues = set(ligand_array.res_name)
print(f"Isolated ligand residue(s): {unique_residues}")
print(f"Total ligand atoms: {len(ligand_array)}")

Isolated ligand residue(s): {np.str_('AQ4')}
Total ligand atoms: 29


In [4]:
import urllib.request

# Download a second structure
urllib.request.urlretrieve("https://files.rcsb.org/download/4HJO.pdb", "4hjo.pdb")
target_array = bsio.load_structure("4hjo.pdb", extra_fields=["charge"])
print("Loaded second structure! Total atoms:", len(target_array))

Loaded second structure! Total atoms: 2264


In [5]:
# Filter for alpha carbon atoms (CA) that are part of standard amino acids
ca1 = source_array[(source_array.atom_name == "CA") & struc.filter_amino_acids(source_array)]
ca2 = target_array[(target_array.atom_name == "CA") & struc.filter_amino_acids(target_array)]

print(f"Backbone atoms - Structure 1: {len(ca1)}, Structure 2: {len(ca2)}")

Backbone atoms - Structure 1: 312, Structure 2: 278


In [6]:
import numpy as np
import biotite.structure as struc

# Find common residue IDs present in both structures
common_res = np.intersect1d(ca1.res_id, ca2.res_id)

# Filter both arrays to only include those matching positions
ca1_matched = ca1[np.isin(ca1.res_id, common_res)]
ca2_matched = ca2[np.isin(ca2.res_id, common_res)]

print(f"Matched backbone atoms for alignment: {len(ca1_matched)}")

Matched backbone atoms for alignment: 278


In [7]:
# Check lengths
print(f"Lengths -> ca1: {len(ca1_matched)}, ca2: {len(ca2_matched)}")

# Slice both to the exact same length to prevent shape broadcasting errors
min_len = min(len(ca1_matched), len(ca2_matched))
ca1_aligned_fixed = ca1_matched[:min_len]
ca2_aligned_mobile = ca2_matched[:min_len]

# Now calculate the superimposition (fixed, mobile)
rot, trans = struc.superimpose(ca1_aligned_fixed, ca2_aligned_mobile)

Lengths -> ca1: 278, ca2: 278


In [8]:
# 1. Calculate superimposition on the matched C-alphas 
# struc.superimpose returns (aligned_mobile, transformation_object)
_, transform = struc.superimpose(fixed=ca2_matched, mobile=ca1_matched)

# 2. Apply that exact transformation to the full source array
aligned_source = transform.apply(source_array)

print("Alignment completed successfully!")

Alignment completed successfully!


In [9]:
# Re-run superimpose to capture both the aligned mobile array and the transformation
aligned_ca1, transform = struc.superimpose(fixed=ca2_matched, mobile=ca1_matched)

# Calculate the RMSD between the aligned structure and the reference structure
rmsd_value = struc.rmsd(aligned_ca1, ca2_matched)
print(f"Backbone RMSD between 1M17 and 4HJO: {rmsd_value:.2f} Å")

Backbone RMSD between 1M17 and 4HJO: 4.08 Å


In [10]:
import biotite.structure.io as bsio

# Save the aligned source array to a PDB file
bsio.save_structure("aligned_1m17.pdb", aligned_source)
print("Aligned structure saved as 'aligned_1m17.pdb'!")

Aligned structure saved as 'aligned_1m17.pdb'!


In [11]:
import sys

# Add the path where py3dmol is installed to Python's search path
path = '/opt/anaconda3/lib/python3.14/site-packages'
if path not in sys.path:
    sys.path.append(path)

In [12]:
# 1. Isolate heteroatoms (ligands/waters) from your full source array (1M17)-use this to extract ligand after the protein has been aligned.
ligand_mask = source_array.hetero
mobile_ligand = source_array[ligand_mask]

# 2. Apply the exact same transformation matrix used for the protein
aligned_ligand = transform.apply(mobile_ligand)

print(f"Extracted and aligned ligand atoms: {len(aligned_ligand)}")

Extracted and aligned ligand atoms: 49


In [15]:
import biotite.structure.io as bsio

bsio.save_structure("aligned_ligand.pdb", aligned_ligand)
print("Aligned ligand saved as 'aligned_ligand.pdb'!")

Aligned ligand saved as 'aligned_ligand.pdb'!


In [20]:
from pathlib import Path
import biotite.structure.io as bsio

# 1. Delete the incorrect file
file_path = Path("taligned_ligand.pdb")
if file_path.exists():
    file_path.unlink()
    print("Deleted incorrect taligned_ligand.pdb")

# 2. Extract the correct reference ligand (no transform!); use this to extract ligand before the protein has been aligned.
reference_ligand = target_array[target_array.hetero]

# 3. Save it correctly
bsio.save_structure("taligned_ligand.pdb", reference_ligand)
print("Successfully saved correct reference ligand!")

Deleted incorrect taligned_ligand.pdb
Successfully saved correct reference ligand!


In [22]:
!pip install nglview

In [23]:
import nglview as nv
import biotite.structure.io as bsio

# 1. Save your protein target array to a PDB file
bsio.save_structure("protein_target.pdb", target_array)

# 2. Load the protein into NGLView
view = nv.show_file("protein_target.pdb")

# 3. Add your aligned ligands as components
view.add_component("aligned_ligand.pdb")
view.add_component("taligned_ligand.pdb")

# Display the interactive viewer
display(view)

NGLWidget()